# Shared initialization across QASP

This notebook constructs and executes small models from each circuit-producing QASP module.
Every code cell contains assertions; the final table is built only from completed checks.

```python
from qasp.initialization import Normal, Uniform, Constant
config = {"init": Normal(mu=0.25, std=0.05),
          "input_scaling_init": Uniform(low=0.8, high=1.2)}
```

`init` controls ansatz parameters. `input_scaling_init` controls the full input-scale/projection
array, including all data-reuploading layers. `Normal.std` is a standard deviation.
Specifications are immutable and reusable; each model draws fresh values. Classical heads retain
their existing initialization. Legacy strings remain supported.

Defaults are preserved: scaled Torch layers use `init` for both groups when the scale override
is omitted; PQCBM scales default to ones. Explicit arrays override their own parameter group.
QAOA/PCE accept `init` at construction and an override in `solve(initial_parameters=...)`.
LR-QAOA has a deterministic ramp instead of a trainable random start.

Run all cells from the repository root or `examples/` using an environment with QASP's
Python dependencies and Jupyter installed. All circuits are small and run locally; no training
of an RL agent, remote service, MPI cluster, or GPU is needed.

For the project environment, run `conda activate torchlane` and launch Jupyter from
that shell. Select its Python kernel; the first cell prints the executable and versions.

In [ ]:
%matplotlib inline

from pathlib import Path
import sys
root = Path.cwd()
if not (root / "qasp").is_dir():
    root = root.parent
assert (root / "qasp").is_dir(), "Run from the repository root or examples/"
sys.path.insert(0, str(root))
sys.dont_write_bytecode = True

from copy import deepcopy
import numpy as np
import pennylane as qml
import torch
from IPython.display import Markdown, display
from qasp.initialization import (
    Normal, Uniform, Constant, Zeros, Ones, Choice, Beta, TruncatedNormal,
    LogNormal, Gamma, Exponential, Laplace, Rademacher,
)
from qasp import (
    AngleEmbedding, NeuralEmbedding, AngleAnsatz, IQPAnsatz, CircuitBuilder,
    create_classifier, create_kernel, QFKernel, create_qcbm,
    QAOA, WarmStartQAOA, PCE, RecursiveQAOA, LRQAOA, QUBO,
    barren_plateau_diagnoser,
)
from qasp.rl import create_policy, create_qfunction

torch.manual_seed(7)
np.random.seed(7)
checks = []
def passed(module, detail):
    checks.append((module, detail))
    print(f"PASS: {module} — {detail}")

print(f"Python {sys.version.split()[0]}, PennyLane {qml.__version__}, Torch {torch.__version__}")
print(f"Python executable: {sys.executable}")

## Distribution contracts

NumPy sampling accepts an explicit `numpy.random.Generator`. Calling an initializer on a
Torch tensor fills it in place and accepts an optional `torch.Generator`, preserving dtype,
device and leaf-parameter status. CPU tests are used here. NumPy and Torch RNG algorithms
are different; equal seeds reproduce results within a backend, not across backends.

In [ ]:
normal = Normal(mu=-0.4, std=0.15)
uniform = Uniform(low=0.8, high=1.2)
a = normal.sample((50_000,), rng=np.random.default_rng(10))
b = normal.sample((50_000,), rng=np.random.default_rng(10))
np.testing.assert_array_equal(a, b)
assert abs(a.mean() + 0.4) < 0.005
assert abs(a.std() - 0.15) < 0.005
scales = uniform.sample((50_000,), rng=np.random.default_rng(11))
assert scales.min() >= 0.8 and scales.max() < 1.2

tensor = torch.empty(100, dtype=torch.float64, requires_grad=True)
assert normal(tensor, generator=torch.Generator().manual_seed(10)) is tensor
assert tensor.dtype == torch.float64 and tensor.is_leaf and tensor.requires_grad
assert set(Choice((0, np.pi / 2)).sample(1000)) == {0, np.pi / 2}
np.testing.assert_array_equal(Zeros().sample((2, 2)), np.zeros((2, 2)))
np.testing.assert_array_equal(Ones().sample((2, 2)), np.ones((2, 2)))
passed("initialization", "arbitrary means/ranges, shape, dtype, deterministic seeds, and discrete values")

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(9, 2.6))
axes[0].hist(a, bins=50, color="#4b88b8", density=True)
axes[0].set(title="Ansatz: Normal(-0.4, 0.15)", xlabel="parameter", ylabel="density")
axes[1].hist(scales, bins=40, color="#8f75af", density=True)
axes[1].set(title="Input scales: Uniform(0.8, 1.2)", xlabel="scale")
fig.tight_layout()
plt.show()

## More distributions and suggested starting points

For ansatz angles, compare `Normal(0, 0.1)` with
`TruncatedNormal(0, 0.1, low=-0.2, high=0.2)`. The latter samples a conditioned
normal rather than clipping a normal sample. Bounds are actual parameter values;
`mu` and `std` describe the normal before truncation.

For **AngleEmbedding input scales**, start with `Ones()` or
`Beta(5, 5, low=0.8, high=1.2)`. A symmetric Beta with shapes above one
concentrates near the midpoint. Shapes below one favor the endpoints;
unequal shapes produce skew. For **NeuralEmbedding projection weights**, a
centered `Normal(0, 1 / np.sqrt(n_features))` is a useful baseline when features
are standardized and roughly independent. These are empirical starting points,
not a universal best choice or a guarantee against barren plateaus.

| Distribution | Parameter convention / use |
| --- | --- |
| `Beta(alpha, beta, low, high)` | Bounded, adjustable concentration and skew |
| `TruncatedNormal(mu, std, low, high)` | Gaussian restricted to a finite interval |
| `LogNormal(mu, std)` | Log-space mean/std; use `mu=-std**2/2` for arithmetic mean one |
| `Gamma(concentration, rate)` | Positive; mean `concentration/rate`, variance `concentration/rate**2` |
| `Exponential(rate)` | Positive, concentrated near zero; mean `1/rate` |
| `Laplace(mu, scale)` | Symmetric with heavier tails; std `sqrt(2)*scale` |
| `Rademacher(scale)` | Equal probability of `-scale` or `+scale` |

Beta, Gamma, Laplace and TruncatedNormal fill Torch tensors through a local CPU
sampler seeded from Torch's RNG, then copy to the target dtype/device. Explicit
generators leave both global RNG states untouched. Other built-ins use native
Torch fills. All produce ordinary trainable parameters.

Definitions: [NumPy Beta](https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.beta.html),
[SciPy truncation conventions](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.truncnorm.html),
[PyTorch distributions](https://docs.pytorch.org/docs/stable/distributions.html).

In [ ]:
strategies = {
    "Beta": Beta(5, 5, low=0.8, high=1.2),
    "TruncatedNormal": TruncatedNormal(0, 0.1, low=-0.2, high=0.2),
    "LogNormal": LogNormal(-0.005, 0.1),
    "Gamma": Gamma(concentration=100, rate=100),
    "Exponential": Exponential(rate=10),
    "Laplace": Laplace(mu=0, scale=0.1),
    "Rademacher": Rademacher(scale=0.1),
}
samples = {}
for name, spec in strategies.items():
    values = spec.sample(20_000, rng=np.random.default_rng(20))
    np.testing.assert_array_equal(values, spec.sample(20_000, rng=np.random.default_rng(20)))
    tensor = spec(torch.empty(100, dtype=torch.float64, requires_grad=True),
                  generator=torch.Generator().manual_seed(20))
    repeat = spec(torch.empty_like(tensor), generator=torch.Generator().manual_seed(20))
    torch.testing.assert_close(tensor, repeat)
    assert tensor.is_leaf and tensor.requires_grad and torch.isfinite(tensor).all()
    assert values.shape == (20_000,) and np.isfinite(values).all()
    samples[name] = values
    passed("initialization", f"{name}: reproducible NumPy/Torch sampling and trainable tensors")

assert np.all((samples["Beta"] >= 0.8) & (samples["Beta"] <= 1.2))
assert np.all((samples["TruncatedNormal"] > -0.2) & (samples["TruncatedNormal"] < 0.2))
for name in ("Beta", "LogNormal", "Gamma"):
    assert abs(samples[name].mean() - 1) < 0.01
for name in ("LogNormal", "Gamma", "Exponential"):
    assert samples[name].min() > 0
assert set(samples["Rademacher"]) == {-0.1, 0.1}

fig, axes = plt.subplots(2, 3, figsize=(11, 5.8))
for ax, name in zip(axes.flat, list(strategies)[:6]):
    ax.hist(samples[name], bins=45, density=True, color="#527d9b", alpha=0.85)
    ax.set(title=name, xlabel="initial value", ylabel="density")
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle("Six continuous initialization distributions", fontsize=14)
fig.tight_layout()
plt.show()
print("Rademacher: equally likely -0.1 and +0.1 (discrete, not a continuous density).")

## Shared configuration and assertions

A zero-width normal makes the ansatz check exact; a nonzero uniform range checks that input
scales are sampled separately. The shape assertions below also cover AngleEmbedding,
NeuralEmbedding and repeated data encoding. Gradient checks require finite, nonzero gradients
for both trainable groups, not just successful construction.

In [ ]:
def configuration(ansatz="AngleAnsatz", feature_map="AngleEmbedding"):
    ac = {"n_layers": 2, "reuploading": True}
    if ansatz == "AngleAnsatz":
        ac.update(rotations="YZ", include_imprimitive=True,
                  imprimitive="CNOT", imprimitive_pattern="chain")
    else:
        ac["imprimitive_pattern"] = "chain"
    fm = {"features": 2, "rotations": "Y"}
    if feature_map == "AngleEmbedding":
        fm["input_scaling"] = True
    return {
        "ansatz": ansatz, "ansatz_config": ac,
        "feature_map": feature_map, "feature_map_config": fm,
        "init": Normal(0.25, 0), "input_scaling_init": Uniform(0.8, 1.2),
        "device": "default.qubit", "diff_method": "backprop",
    }

def array(value):
    return value.detach().cpu().numpy() if isinstance(value, torch.Tensor) else np.asarray(value)

def check_groups(params, scales):
    np.testing.assert_allclose(array(params), 0.25)
    assert np.all((array(scales) >= 0.8) & (array(scales) <= 1.2))

def check_gradients(loss, parameters):
    grads = torch.autograd.grad(loss, list(parameters))
    assert all(torch.isfinite(g).all() and g.abs().sum() > 1e-9 for g in grads)

x = torch.tensor([0.2, -0.3])

## Circuits and feature maps

The lower-level builder retains its `initialization=` keyword; its new `input_scaling_init=`
argument has the same meaning as in model dictionaries. Each example evaluates a real QNode.

In [ ]:
for ansatz_class in (AngleAnsatz, IQPAnsatz):
    for embedding_class in (AngleEmbedding, NeuralEmbedding):
        extra = {"input_scaling": True} if embedding_class is AngleEmbedding else {}
        embedding = embedding_class(n_qubits=2, features=2, rotations="Y", **extra)
        ac = dict(n_qubits=2, n_layers=2, feature_map=embedding, reuploading=True)
        if ansatz_class is AngleAnsatz:
            ac["rotations"] = "YZ"
        builder = CircuitBuilder(n_qubits=2)
        builder.set_ansatz(ansatz_class(**ac))
        builder.set_measurement("expectation", observables=["Z 0", "Z 1"])
        builder.build()
        builder.compile(device="default.qubit", diff_method="backprop")
        layer = builder.create_torchlayer(
            initialization=Normal(0.25, 0), input_scaling_init=Uniform(0.8, 1.2))
        check_groups(layer.params, layer.input_scaling_params)
        assert layer.input_scaling_params.numel() == (4 if embedding_class is AngleEmbedding else 8)
        check_gradients(layer(x).square().sum(), layer.parameters())
        passed("circuits / feature_maps", f"{ansatz_class.__name__} + {embedding_class.__name__}")

## Classifiers

Both Born-probability and expectation readouts use the same initialization interface.

In [ ]:
for readout in ("probabilities", "expectation"):
    cfg = configuration()
    cfg["readout"] = readout
    if readout == "expectation":
        cfg["observables"] = ["Z 0", "Z 1"]
    classifier = create_classifier(cfg, n_qubits=2, n_classes=2)
    check_groups(classifier.qlayer.params, classifier.qlayer.input_scaling_params)
    check_gradients(classifier(x).reshape(-1)[0], classifier.qlayer.parameters())
    passed("classifiers", readout + " readout, both trainable parameter groups")

## RL policies and PQDQN

Only quantum ansatz and encoding weights use these settings. Policy/Q-value heads preserve
their existing initialization. The same construction API works with the differentiable
Pauli-propagation backend for supported expectation models.

In [ ]:
for kind in ("PQCBorn", "PQCSoftmax"):
    cfg = configuration()
    cfg["n_actions"] = 2
    if kind == "PQCBorn":
        cfg.update(policy="parity-like", parity_cnot_decomposition=False)
    else:
        cfg.update(observables=["Z 0", "Z 1"], fc_layer=True)
    policy = create_policy(kind, deepcopy(cfg), n_qubits=2)
    check_groups(policy.qlayer.params, policy.qlayer.input_scaling_params)
    check_gradients(policy(x).reshape(-1)[0], policy.qlayer.parameters())
    passed("rl.policies", kind)

for backend in ("default.qubit", "lightning.pp"):
    cfg = configuration()
    cfg.update(device=backend, observables=["Z 0", "Z 1"],
               pp_truncation=None, pp_topM=None, pp_tau=0, pp_prune=False)
    qfunction = create_qfunction(cfg, n_qubits=2, n_actions=2)
    check_groups(qfunction.qlayer.params, qfunction.qlayer.input_scaling_params)
    check_gradients(qfunction(x).reshape(-1)[0], qfunction.qlayer.parameters())
    passed("rl.qfunctions", "PQDQN on " + backend)

## Quantum kernels

Trainable global fidelity, localized fidelity, and projected kernels initialize both groups.
Encoding-only kernels can draw fixed input scales once; they still have no trainable ansatz.
The legacy Fisher kernel accepts explicit parameters, produced here with `.sample`.

In [ ]:
for kind in ("fidelity", "localized", "projected"):
    cfg = configuration()
    cfg["kernel_type"] = "projected" if kind == "projected" else "fidelity"
    if kind == "localized":
        cfg["projectors"] = ["P 0"]
    kernel = create_kernel(cfg, n_qubits=2)
    check_groups(kernel.qlayer.params, kernel.qlayer.input_scaling_params)
    gram = kernel.gram([[0.2, -0.3], [0.7, 0.4]], return_tensor=True)
    assert gram.shape == (2, 2)
    check_gradients(gram[0, 1], kernel.parameters())
    passed("qkernels", "trainable " + kind)

fixed_cfg = configuration()
fixed_cfg.pop("ansatz")
fixed_cfg.pop("ansatz_config")
fixed_cfg["input_scaling_init"] = Constant(1.2)
fixed = create_kernel(fixed_cfg, n_qubits=2)
np.testing.assert_allclose(fixed._fixed_isp, 1.2)
assert fixed.parameters() == []
assert np.isfinite(fixed.gram([[0.2, 0.3], [0.8, -0.2]])).all()
passed("qkernels", "encoding-only fixed input scales")

@qml.qnode(qml.device("default.qubit", wires=2), interface="autograd")
def fisher_circuit(params):
    qml.RY(params[0], wires=0)
    qml.RX(params[1], wires=1)
    return qml.probs(wires=[0, 1])

fisher_params = qml.numpy.array(Uniform(0.2, 0.8).sample(2, rng=np.random.default_rng(8)),
                                requires_grad=True)
fisher = QFKernel(fisher_circuit, fisher_params, fisher="classical")
fisher_gram = fisher.gram([[0, 0], [1, 1]])
assert fisher_gram.shape == (2, 2) and np.isfinite(fisher_gram).all()
passed("qkernels", "QFKernel with explicitly sampled parameters")

## Generative learning: Torch and Autograd

PQCBM's `seed` controls the built-in distributions using its local NumPy RNG on both
interfaces. One optimization step verifies that independently initialized scales and
ansatz weights remain trainable. Without the scale override, scales still start at one.

In [ ]:
data = np.array([[0, 0], [1, 1], [0, 0], [1, 1]])
for interface in ("torch", "autograd"):
    cfg = {**configuration(), "inputs": [0.2, -0.3], "interface": interface,
           "seed": 12, "lr": 0.01}
    model = create_qcbm(cfg, n_qubits=2, data=data)
    same = create_qcbm(cfg, n_qubits=2, data=data)
    check_groups(model.params, model.input_scaling_params)
    np.testing.assert_array_equal(array(model.input_scaling_params), array(same.input_scaling_params))
    before_params, before_scales = array(model.params).copy(), array(model.input_scaling_params).copy()
    _, history = model.train(1)
    assert np.isfinite(history).all()
    assert not np.array_equal(array(model.params), before_params)
    assert not np.array_equal(array(model.input_scaling_params), before_scales)
    passed("genlearn", f"PQCBM {interface}: reproducibility and training both groups")

cfg.pop("input_scaling_init")
default_model = create_qcbm(cfg, n_qubits=2, data=data)
np.testing.assert_allclose(array(default_model.input_scaling_params), 1.)
passed("genlearn", "legacy default input scales remain ones")

## Optimization: QAOA, warm starts, multi-angle QAOA and PCE

These algorithms do not have input feature maps, so their initializer controls the quantum
angle vector. Their `seed` controls built-in distribution sampling. The first solve uses an
identity optimizer intentionally: its returned parameters prove which initial vector reached
the optimizer. A separate Torch solve performs a real Adam step.

In [ ]:
problem = QUBO.maxcut(3, [(0, 1), (1, 2), (0, 2)])
factories = {
    "QAOA": lambda **kw: QAOA(problem, n_layers=2, **kw),
    "multi-angle QAOA": lambda **kw: QAOA(problem, n_layers=2, parametrization="multi_angle", **kw),
    "WarmStartQAOA": lambda **kw: WarmStartQAOA(problem, warm_start=[0.2, 0.6, 0.7], **kw),
    "PCE": lambda **kw: PCE(problem, **kw),
}
for name, factory in factories.items():
    first = factory(init=Uniform(-0.2, 0.4), seed=5)
    second = factory(init=Uniform(-0.2, 0.4), seed=5)
    start = first.initial_parameters()
    np.testing.assert_array_equal(start, second.initial_parameters())
    assert np.all((start >= -0.2) & (start <= 0.4))
    result = first.solve(initial_parameters=Constant(0.3),
                         optimizer=lambda objective, values, **kw: values)
    np.testing.assert_allclose(result.parameters, 0.3)
    assert np.isfinite(result.energy)
    torch_solver = factory(init=Normal(0.25, 0.05), seed=5, backend="torch")
    trained = torch_solver.solve(optimizer="Adam", optimizer_options={"max_steps": 1, "lr": 0.01})
    assert np.isfinite(trained.parameters).all() and np.isfinite(trained.energy)
    passed("optimization", name + ": seeded creation, solve override, Torch update")

## Recursive and deterministic optimization

RQAOA forwards its initializer to every reduced QAOA subproblem. LR-QAOA deliberately keeps
its fixed ramp; passing `init` is rejected so it cannot be silently ignored.

In [ ]:
recursive = RecursiveQAOA(problem, cutoff=2, init=Constant(0.2),
                           optimizer=lambda objective, values, **kw: values)
recursive_result = recursive.solve()
assert len(recursive.step_results) == 1 and np.isfinite(recursive_result.energy)
np.testing.assert_allclose(recursive.step_results[0].parameters, 0.2)
passed("optimization", "RQAOA forwards the initializer to reduced subproblems")

ramp = LRQAOA(problem, n_layers=2, delta_gamma=0.6, delta_beta=0.3)
np.testing.assert_allclose(ramp.solve().parameters.reshape(2, 2), ramp.ramp_parameters)
try:
    LRQAOA(problem, init=Normal())
except ValueError as error:
    assert "fixed ramp" in str(error)
else:
    raise AssertionError("A fixed ramp must not silently accept a random initializer")
passed("optimization", "LR-QAOA fixed-ramp contract")

## Diagnostics

The concentration diagnoser uses a local seeded Torch generator for built-in initializer
objects. It can resample input scales independently and restores the model afterwards.
Custom tensor callables remain supported but own their randomness.

In [ ]:
model = create_classifier(configuration(), n_qubits=2, n_classes=2)
snapshot = {key: value.clone() for key, value in model.qlayer.state_dict().items()}
observed = []
def cost(model, state):
    assert torch.all(model.qlayer.input_scaling_params == 1.5)
    observed.append(model.qlayer.params.detach().clone())
    return model(state).reshape(-1)[0]

settings = dict(n_samples=5, initialization=Normal(0.2, 0.1),
                input_scaling_init=Constant(1.5), state=x,
                cost_function=cost, seed=19, return_samples=True)
global_rng = torch.get_rng_state().clone()
first = barren_plateau_diagnoser(model, **settings)
second = barren_plateau_diagnoser(model, **settings)
assert first["variance"] == second["variance"]
assert not torch.equal(observed[0], observed[1])
torch.testing.assert_close(observed[0], observed[5])
assert torch.equal(global_rng, torch.get_rng_state())
for name, before in snapshot.items():
    torch.testing.assert_close(model.qlayer.state_dict()[name], before)
passed("diagnostics", "independent groups, seeded draws, restoration and unchanged global RNG")

## Explicit weights, compatibility and validation

Explicit arrays independently override the relevant initializer. Legacy names map to the
same distributions; misspellings now raise instead of silently selecting a fallback.
`Choice((0, np.pi/2))` is the recommended replacement for the historical discrete alias:
the builder's old alias sampled 0/pi, whereas diagnostics used 0/pi/2. Those old meanings
are preserved at their original entry points.

In [ ]:
# Reuse the manual builder from the circuits section.
shape_scales, shape_params = builder.ansatz.get_param_shapes()
layer = builder.create_torchlayer(
    params=torch.full(shape_params, 0.6),
    initialization=Zeros(), input_scaling_init=Constant(1.3))
np.testing.assert_allclose(array(layer.params), 0.6)
np.testing.assert_allclose(array(layer.input_scaling_params), 1.3)
builder.set_weights(input_scaling_params=torch.ones(shape_scales))
np.testing.assert_allclose(array(layer.params), 0.6)
np.testing.assert_allclose(array(layer.input_scaling_params), 1.0)
passed("circuits", "partial explicit overrides and partial weight updates")

cfg = configuration()
cfg.pop("input_scaling_init")
torch.manual_seed(27)
old = create_classifier({**cfg, "init": "small_normal"}, 2, 2)
torch.manual_seed(27)
new = create_classifier({**cfg, "init": Normal(0, 0.1)}, 2, 2)
for key in old.qlayer.state_dict():
    torch.testing.assert_close(old.qlayer.state_dict()[key], new.qlayer.state_dict()[key])

def custom_fill(tensor):
    tensor.fill_(0.4)
custom = create_classifier({**configuration(), "init": custom_fill}, 2, 2)
np.testing.assert_allclose(array(custom.qlayer.params), 0.4)
for invalid in (lambda: Normal(0, -1), lambda: Uniform(2, 1),
                lambda: create_classifier({**cfg, "init": "normla_0_1"}, 2, 2)):
    try:
        invalid()
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid initialization was accepted")
passed("compatibility", "legacy aliases, custom callables, and validation errors")

## Bounded initializers across model families

The same TruncatedNormal/Beta pair works for both parameter groups without
changing the model factory. QAOA/PCE use the same distributions for their angle
vectors; the diagnoser can resample each group independently.

In [ ]:
for family in ("classifier", "policy", "pqdqn", "kernel", "pqcbm"):
    cfg = configuration()
    cfg.update(init=TruncatedNormal(0.2, 0.1, 0.05, 0.4),
               input_scaling_init=Beta(5, 5, 0.8, 1.2))
    if family == "classifier":
        model = create_classifier(cfg, 2, 2)
    elif family == "policy":
        cfg.update(n_actions=2, observables=["Z 0", "Z 1"])
        model = create_policy("PQCSoftmax", cfg, 2)
    elif family == "pqdqn":
        model = create_qfunction({**cfg, "observables": ["Z 0", "Z 1"]}, 2, 2)
    elif family == "kernel":
        model = create_kernel(cfg, 2)
    else:
        model = create_qcbm({**cfg, "inputs": [0.2, -0.3], "seed": 7}, 2, [[0, 0], [1, 1]])
    layer = model if family == "pqcbm" else model.qlayer
    assert torch.all((layer.params >= 0.05) & (layer.params <= 0.4))
    assert torch.all((layer.input_scaling_params >= 0.8) & (layer.input_scaling_params <= 1.2))
    if family == "pqcbm":
        loss = model.loss()
    elif family == "kernel":
        loss = model.gram([[0.2, -0.3], [0.7, 0.4]], return_tensor=True)[0, 1]
    else:
        loss = model(torch.tensor([0.2, -0.3])).reshape(-1)[0]
    grads = torch.autograd.grad(loss, [layer.params, layer.input_scaling_params])
    assert all(torch.isfinite(g).all() and g.abs().sum() > 1e-9 for g in grads)
    passed(family, "TruncatedNormal ansatz + Beta scales: bounds and nonzero gradients")

for family in (QAOA, PCE):
    spec = Beta(2, 3, low=-0.2, high=0.4)
    solver = family(problem, init=spec, seed=5)
    values = solver.initial_parameters()
    np.testing.assert_array_equal(values, family(problem, init=spec, seed=5).initial_parameters())
    assert np.all((values >= -0.2) & (values <= 0.4))
    result = solver.solve(optimizer=lambda objective, x, **kwargs: x)
    assert np.isfinite(result.energy)
    passed("optimization", f"{family.__name__}: bounded Beta angles, reproducibility and solve")

model = create_classifier(configuration(), 2, 2)
state = torch.tensor([0.2, -0.3])
saved = {k: v.clone() for k, v in model.qlayer.state_dict().items()}
kwargs = dict(n_samples=4, initialization=TruncatedNormal(0, 0.1, -0.2, 0.2),
              input_scaling_init=Beta(5, 5, 0.8, 1.2), state=state, seed=5)
a = barren_plateau_diagnoser(model, **kwargs)
b = barren_plateau_diagnoser(model, **kwargs)
assert a["variance"] == b["variance"]
for key, value in saved.items():
    torch.testing.assert_close(model.qlayer.state_dict()[key], value)
passed("diagnostics", "TruncatedNormal/Beta resampling is reproducible and restores weights")

## Completed checks

Each row is added only after its assertions pass. The companion regression tests are in
`tests/test_initialization.py`; the existing module suites additionally check that default
behavior remains compatible. MPI/GPU execution requires separate backend environments;
this notebook validates local CPU statevector and differentiable Pauli-propagation paths.

In [ ]:
assert {"initialization", "circuits / feature_maps", "classifiers", "rl.policies", "rl.qfunctions",
        "qkernels", "genlearn", "optimization", "diagnostics", "circuits", "compatibility"}.issubset(
            {module for module, _ in checks})
rows = "\n".join(f"| {module} | {detail} | PASS |" for module, detail in checks)
display(Markdown("| Module | Check | Result |\n| --- | --- | --- |\n" + rows))
print(f"All {len(checks)} cross-module checks passed.")